# Modeling - MIAX14 Hackathon
Pipeline de entrenamiento, validación y generación de predicciones.

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from features import load_data, build_feature_matrix, INDICES
from models import MultiIndexModel
from predict import predict_autoregressive

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='darkgrid')

ImportError: cannot import name 'MultiIndexModel' from 'models' (/Users/emiliosanchez/Hackaton_30_mayo/hackathon_miax14/notebooks/../src/models.py)

## 1. Cargar y construir features

In [ ]:
data = load_data('../data')
train_feat, test_feat = build_feature_matrix(data)

target_cols = INDICES
feature_cols = [c for c in train_feat.columns if c not in target_cols]

X = train_feat[feature_cols]
y = train_feat[target_cols]

print(f'Features: {len(feature_cols)}')
print(f'Training samples: {len(X)}')

## 2. Train/Validation split (last ~1 year)

In [ ]:
VAL_SIZE = 252
X_train, X_val = X.iloc[:-VAL_SIZE], X.iloc[-VAL_SIZE:]
y_train, y_val = y.iloc[:-VAL_SIZE], y.iloc[-VAL_SIZE:]

print(f'Train: {X_train.index.min()} -> {X_train.index.max()} ({len(X_train)} rows)')
print(f'Val:   {X_val.index.min()} -> {X_val.index.max()} ({len(X_val)} rows)')

## 3. Entrenar modelos LightGBM

In [ ]:
model = MultiIndexModel(INDICES)
model.fit(X_train, y_train, X_val, y_val)
print('Training complete.')

## 4. Evaluación en validación

In [ ]:
val_pred = model.predict(X_val)

def rmse(a, b):
    return float(np.sqrt(((a.values - b.values) ** 2).mean()))

print(f'Global RMSE: {rmse(y_val, val_pred):.2f}')
for col in INDICES:
    r = float(np.sqrt(((y_val[col] - val_pred[col]) ** 2).mean()))
    print(f'  {col}: {r:.2f}')

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
for ax, col in zip(axes.flatten(), INDICES):
    y_val[col].plot(ax=ax, label='Real', lw=0.9)
    val_pred[col].plot(ax=ax, label='Predicción', lw=0.9, linestyle='--')
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Importancia de features

In [ ]:
imp = model.feature_importance()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flatten(), INDICES):
    top = imp[col].sort_values(ascending=False).head(15)
    top.plot(kind='barh', ax=ax, title=f'{col} - Top features')
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Predicción autorregresiva en test

In [ ]:
test_preds = predict_autoregressive(model, data['train_idx'], test_feat)
test_preds.head()

In [ ]:
# Visualizar predicciones vs histórico reciente
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
for ax, col in zip(axes.flatten(), INDICES):
    data['train_idx'][col].iloc[-500:].plot(ax=ax, label='Histórico', lw=0.9)
    test_preds[col].plot(ax=ax, label='Predicción test', lw=0.9, linestyle='--')
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Guardar submission

In [ ]:
out = test_preds.copy()
out.index.name = 'Date'
out = out.reset_index()
out['Date'] = out['Date'].dt.strftime('%Y-%m-%d')
out.to_excel('../submissions/submission.xlsx', index=False)
print('Submission saved.')
out